# Fine-tune Qwen2.5-1.5B-Instruct bằng Unsloth (LoRA)
Notebook này hướng dẫn cách sử dụng Unsloth để fine-tune mô hình Qwen2.5-1.5B-Instruct cho tập dữ liệu VinMMRC 2.0 (đọc hiểu trắc nghiệm).

In [2]:
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo -y

Found existing installation: torch 2.12.1
Uninstalling torch-2.12.1:
  Successfully uninstalled torch-2.12.1
Found existing installation: torchvision 0.27.1
Uninstalling torchvision-0.27.1:
  Successfully uninstalled torchvision-0.27.1
Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0
Found existing installation: xformers 0.0.35
Uninstalling xformers-0.0.35:
  Successfully uninstalled xformers-0.0.35
Found existing installation: transformers 5.5.0
Uninstalling transformers-5.5.0:
  Successfully uninstalled transformers-5.5.0
Found existing installation: trl 0.24.0
Uninstalling trl-0.24.0:
  Successfully uninstalled trl-0.24.0
Found existing installation: unsloth 2026.6.9
Uninstalling unsloth-2026.6.9:
  Successfully uninstalled unsloth-2026.6.9
Found existing installation: unsloth_zoo 2026.6.7
Uninstalling unsloth_zoo-2026.6.7:
  Successfully uninstalled unsloth_zoo-2026.6.7


In [3]:
# Bước 2: Cài đặt lại bản PyTorch 2.6 ổn định, hỗ trợ CUDA 12.4 (hoàn toàn tương thích ngược với CUDA 12.8 của máy)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 59.8 MB/s  0:00:10:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 62.5 MB/s  0:00:04:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 62.9 MB/s  0:00:04:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 62.6 MB/s  0:00:02ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 62.6 MB/s  0:00:09:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 97.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 62.4 MB/s  0:00:03:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 63.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 62.5 MB/s  0:00:01m0:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 62.5 MB/s 

In [3]:
# Bước 3: Cài đặt đồng bộ tất cả các thư viện vệ tinh cốt lõi (Không chỉ định phiên bản lỗi thời)
!pip install transformers trl peft accelerate bitsandbytes xformers --no-cache-dir

# Bước 4: Cài đặt lại bản Unsloth mới nhất từ nhánh chính để nhận diện chính xác card của bạn
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-cache-dir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 59.1 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 66.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 66.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 418.1 MB/s  0:00:00
  Attempting uninstall: datasets━━━━━━━━━━━━━━━━ 0/4 [xformers]
    Found existing installation: datasets 4.3.0m 0/4 [xformers]
    Uninstalling datasets-4.3.0:━━━━━━━━━━━━ 0/4 [xformers]
      Successfully uninstalled datasets-4.3.0 0/4 [xformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [trl]3/4 [trl]sformers]

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-v55_xijr/unsloth_4cdd88a9fade457d81cb1bd60d3b984d
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-v55_xijr/unsloth

In [7]:
import torch


print("--- KIỂM TRA PHIÊN BẢN CÁC THƯ VIỆN ---")
print(f"PyTorch version:     {torch.__version__}")
print(f"CUDA Available:      {torch.cuda.is_available()}")


--- KIỂM TRA PHIÊN BẢN CÁC THƯ VIỆN ---
PyTorch version:     2.11.0+cu128
CUDA Available:      True


In [1]:
# 2. Khởi tạo Mô hình và Tokenizer
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Hỗ trợ tự động RoPE Scaling cho context dài
dtype = None # None để tự động nhận diện (Float16 cho T4/V100, Bfloat16 cho Ampere+)
load_in_4bit = True # Dùng 4bit quantization để tiết kiệm bộ nhớ GPU

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Thiết lập cấu hình LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060 Ti. Num GPUs = 1. Max memory: 7.631 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [2]:
# 3. Tiền xử lý dữ liệu
import json
from datasets import Dataset

# Tải dữ liệu từ file JSON. (Đảm bảo file sample_train.json nằm cùng thư mục)
with open("train_vimmrc.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

formatted_data = []

for item in raw_data:
    article = item.get("article", "")
    questions = item.get("questions", [])
    options_list = item.get("options", [])
    answers = item.get("answers", [])

    # Tạo prompt tách biệt cho mỗi câu hỏi để mô hình dễ học
    for q, opts, ans in zip(questions, options_list, answers):
        letters = ["A", "B", "C", "D", "E", "F", "G", "H"]
        options_text = ""
        for i, opt in enumerate(opts):
            if i < len(letters):
                options_text += f"{letters[i]}. {opt}\n"

        prompt = f"Đọc đoạn văn sau và trả lời câu hỏi bằng cách chọn đáp án đúng nhất.\n\nĐoạn văn:\n{article}\n\nCâu hỏi: {q}\nCác đáp án:\n{options_text}"
        response = f"{ans}" # Mô hình học cách trả lời trực tiếp A, B, C hoặc D

        formatted_data.append({
            "instruction": prompt,
            "output": response
        })

dataset = Dataset.from_list(formatted_data)
print(f"Tổng số mẫu huấn luyện (tính theo từng câu hỏi): {len(dataset)}")
print("\nVí dụ 1 mẫu dữ liệu:\n", dataset[0])

Tổng số mẫu huấn luyện (tính theo từng câu hỏi): 3600

Ví dụ 1 mẫu dữ liệu:
 {'instruction': 'Đọc đoạn văn sau và trả lời câu hỏi bằng cách chọn đáp án đúng nhất.\n\nĐoạn văn:\nMỗi năm có bốn mùa: Xuân, Hạ, Thu, Đông. Mùa Xuân tiết trời ấm áp, cây cối đâm chồi nảy lộc. Mùa Hạ nóng bức, ve sầu kêu inh ỏi. Thu đến, bầu trời trong xanh mát mẻ. Đông về rét ơi là rét.\n\nCâu hỏi: Mỗi năm có mấy mùa?\nCác đáp án:\nA. Hai mùa.\nB. Một mùa.\nC. Bốn mùa.\nD. Ba mùa.\n', 'output': 'C'}


In [3]:
# 4. Format prompt theo định dạng chuẩn của ChatML (dành cho Qwen2.5 Instruct)
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        messages = [
            {"role": "system", "content": "Bạn là một trợ lý AI thông minh, giỏi tiếng Việt và có khả năng đọc hiểu văn bản cực kỳ chính xác."},
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": output}
        ]
        # Áp dụng template, tokenize = False vì Trainer sẽ lo phần tokenize
        text = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = False)
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True,)

In [8]:
# 5. Cấu hình và Bắt đầu Huấn luyện (Fine-Tuning)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import sys
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,     # Tên biến dataset của bạn
    dataset_text_field = "text", # Tên trường text trong dataset của bạn
    max_seq_length = max_seq_length,
    dataset_num_proc = 1,        # Ép tiến trình xử lý data về 1 để NÉ HOÀN TOÀN lỗi pickle
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # max_steps = 60, # Tăng số bước hoặc thay bằng num_train_epochs để train toàn bộ dataset (VD: num_train_epochs = 3)
        # num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to="none", # Nếu không dùng wandb thì để none
        

    ),
)
# Lấy class thực tế đang được dùng trong bộ nhớ
real_config_cls = type(trainer.args)
real_trainer_cls = type(trainer)
# Ép sys.modules trỏ đúng vào module chứa các class này
sys.modules['trl.trainer.sft_config'] = sys.modules[real_config_cls.__module__]
sys.modules['trl.trainer.sft_trainer'] = sys.modules[real_trainer_cls.__module__]
sys.modules[real_config_cls.__module__].SFTConfig = real_config_cls
sys.modules[real_trainer_cls.__module__].SFTTrainer = real_trainer_cls

# Bắt đầu train!
trainer_stats = trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,600 | Num Epochs = 3 | Total steps = 1,350
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
10,0.854187
20,0.979208
30,1.056194
40,0.995676
50,0.924207
60,0.939425
70,0.940653
80,0.856379
90,0.816928
100,0.853392


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1350/tokenizer_config.json.


In [9]:
# 6. Kiểm tra Inference sau khi huấn luyện
FastLanguageModel.for_inference(model) # Kích hoạt chế độ sinh token nhanh của Unsloth x2 tốc độ

messages = [
    {"role": "system", "content": "Bạn là một trợ lý AI thông minh, giỏi tiếng Việt và có khả năng đọc hiểu văn bản cực kỳ chính xác."},
    {"role": "user", "content": dataset[0]["instruction"]},
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=64, use_cache=True)

print("\n--- KẾT QUẢ SINH TỪ MODEL ĐÃ FINE-TUNE ---\n")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mas


--- KẾT QUẢ SINH TỪ MODEL ĐÃ FINE-TUNE ---

system
Bạn là một trợ lý AI thông minh, giỏi tiếng Việt và có khả năng đọc hiểu văn bản cực kỳ chính xác.
user
Đọc đoạn văn sau và trả lời câu hỏi bằng cách chọn đáp án đúng nhất.

Đoạn văn:
Mỗi năm có bốn mùa: Xuân, Hạ, Thu, Đông. Mùa Xuân tiết trời ấm áp, cây cối đâm chồi nảy lộc. Mùa Hạ nóng bức, ve sầu kêu inh ỏi. Thu đến, bầu trời trong xanh mát mẻ. Đông về rét ơi là rét.

Câu hỏi: Mỗi năm có mấy mùa?
Các đáp án:
A. Hai mùa.
B. Một mùa.
C. Bốn mùa.
D. Ba mùa.

assistant
C


/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


In [10]:
# 7. Lưu Model (Chỉ lưu Weights LoRA)
model.save_pretrained("qwen2.5-1.5b-instruct-lora-vinmmrc")
tokenizer.save_pretrained("qwen2.5-1.5b-instruct-lora-vinmmrc")

print("Đã lưu thành công mô hình LoRA!")

# Nếu muốn gộp (merge) LoRA vào model gốc và lưu dạng safetensors thì dùng code sau:
# model.save_pretrained_merged("qwen2.5-1.5b-instruct-vinmmrc-merged", tokenizer, save_method="merged_16bit")

Unsloth: Restored added_tokens_decoder metadata in qwen2.5-1.5b-instruct-lora-vinmmrc/tokenizer_config.json.


Đã lưu thành công mô hình LoRA!


In [ ]:
# =====================================================================
# 8. ĐÁNH GIÁ (EVALUATION) - Accuracy trên tập Test
# =====================================================================
# QUAN TRỌNG: Đặt biến môi trường TRƯỚC KHI import transformers
# để ngăn Unsloth monkey-patch gây lỗi 'apply_qkv'
import os
os.environ["UNSLOTH_IS_OFFLINE"] = "1"

import json
import torch
import re
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# 1. CẤU HÌNH & LOAD MODEL
# ==========================================
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH    = "qwen2.5-1.5b-instruct-lora-vinmmrc"   # Đường dẫn LoRA đã lưu
TEST_DATA_PATH  = "test_vimmrc.json"
VALID_LABELS    = ["A", "B", "C", "D", "E", "F"]

print("Loading Tokenizer và Model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("Load model hoàn tất!")

# ==========================================
# 2. ĐỌC DỮ LIỆU TEST
# ==========================================
with open(TEST_DATA_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

all_samples = []
for item in test_data:
    article = item["article"]
    for i in range(len(item["questions"])):
        if len(item["options"][i]) > 0:
            all_samples.append({
                "article":  article,
                "question": item["questions"][i],
                "options":  item["options"][i],
                "answer":   item["answers"][i],
            })

print(f"\nTổng số câu hỏi test: {len(all_samples)}")

# ==========================================
# 3. HÀM TẠO PROMPT INFERENCE
# ==========================================
SYSTEM_PROMPT = "Bạn là một trợ lý AI giỏi tiếng Việt, có khả năng đọc hiểu và trả lời câu hỏi trắc nghiệm chính xác."

def format_prompt(article, question, options):
    labels = ["A", "B", "C", "D", "E", "F"]
    options_text = "\n".join([f"{labels[i]}. {opt}" for i, opt in enumerate(options)])
    content = (
        f"Đọc đoạn văn sau và trả lời câu hỏi bằng cách chọn đáp án đúng nhất.\n\n"
        f"Đoạn văn:\n{article}\n\n"
        f"Câu hỏi: {question}\n"
        f"Các đáp án:\n{options_text}\n\n"
        f"Đáp án của bạn (chỉ đưa ra một chữ cái A, B, C hoặc D):"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": content},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def extract_label(response: str, valid_labels) -> str:
    """Trích xuất nhãn A/B/C/D từ chuỗi model sinh ra."""
    response = response.strip()
    # Ưu tiên: ký tự đầu tiên nếu là label hợp lệ
    if response and response[0].upper() in valid_labels:
        return response[0].upper()
    # Fallback: tìm pattern 'Đáp án: A' hoặc 'A.' hoặc chữ cái đơn lẻ
    match = re.search(r'\b([A-F])\b', response.upper())
    if match:
        return match.group(1)
    return "X"  # Không xác định được

# ==========================================
# 4. CHẠY INFERENCE + MONITOR
# ==========================================
y_true         = []
y_pred         = []
invalid_preds  = 0
correct_so_far = 0

print(f"\nBắt đầu đánh giá trên {len(all_samples)} câu hỏi...\n")

for idx, sample in enumerate(tqdm(all_samples, desc="Evaluating"), start=1):
    prompt = format_prompt(sample["article"], sample["question"], sample["options"])
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,            # Chỉ cần 1 ký tự (A/B/C/D)
            do_sample=False,             # Greedy decoding
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_length     = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    response         = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    pred_label = extract_label(response, VALID_LABELS)
    true_label = sample["answer"].strip().upper()

    y_true.append(true_label)
    y_pred.append(pred_label)

    is_correct = (pred_label == true_label)
    if pred_label not in VALID_LABELS:
        invalid_preds += 1
    if is_correct:
        correct_so_far += 1

    running_acc = correct_so_far / idx * 100
    tqdm.write(
        f"[{idx}/{len(all_samples)}] "
        f"Q: {sample['question'][:45]}... | "
        f"True: {true_label} | Pred: {pred_label} | "
        f"{'✓' if is_correct else '✗'} | "
        f"Running Acc: {running_acc:.1f}%"
    )

# ==========================================
# 5. TÍNH TOÁN VÀ IN KẾT QUẢ ACCURACY
# ==========================================
accuracy = accuracy_score(y_true, y_pred)

print("\n" + "="*55)
print("   KẾT QUẢ ĐÁNH GIÁ - ViMMRC 2.0")
print("   Model: Qwen2.5-1.5B-Instruct + LoRA (Unsloth)")
print("="*55)
print(f"  Tổng số câu hỏi        : {len(y_true)}")
print(f"  Số câu trả lời đúng    : {correct_so_far}")
print(f"  Số câu lỗi định dạng   : {invalid_preds} (không ra A/B/C/D)")
print(f"  ACCURACY               : {accuracy * 100:.2f}%")
print("="*55)

print("\nCLASSIFICATION REPORT (chi tiết từng nhãn):")
print(classification_report(y_true, y_pred, zero_division=0))

# Lưu kết quả ra file JSON để phân tích sau
results = {
    "accuracy": round(accuracy * 100, 4),
    "total": len(y_true),
    "correct": correct_so_far,
    "invalid_format": invalid_preds,
    "predictions": [
        {"question": s["question"], "true": t, "pred": p, "correct": (t == p)}
        for s, t, p in zip(all_samples, y_true, y_pred)
    ]
}
with open("eval_results_vimmrc.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("\nKết quả chi tiết đã được lưu tại: eval_results_vimmrc.json")
